# Adapter Design Pattern

#### Fitting a Square Peg into a Round Hole.

The Scenario
- The System (Target): A RoundHole that only accepts RoundPegs.
- The Problem (Adaptee): We have a SquarePeg. It doesn't fit directly because the system checks for radius, but the square peg only has width.
- The Solution (Adapter): An adapter that takes the square peg and calculates the equivalent "minimum radius" needed to fit it.

### THE TARGET SYSTEM (Round Hole)

In [1]:
class RoundHole:
    def __init__(self, radius):
        self.radius = radius

    def fits(self, peg) -> bool:
        # The hole expects a peg with a .get_radius() method
        return self.radius >= peg.get_radius()

### THE COMPATIBLE ITEM (Round Peg)

In [2]:
class RoundPeg:
    def __init__(self, radius):
        self.radius = radius

    def get_radius(self):
        return self.radius

### THE INCOMPATIBLE ITEM (Square Peg)

In [5]:
class SquarePeg:
    def __init__(self, width):
        self.width = width

    def get_width(self):
        return self.width
        
    # NOTICE: This class has NO get_radius() method. 
    # It cannot be used with RoundHole directly.

### THE ADAPTER

In [6]:
import math

class SquarePegAdapter:
    """
    This adapter wraps the SquarePeg and pretends to be a RoundPeg.
    """
    def __init__(self, peg: SquarePeg):
        self.peg = peg

    def get_radius(self):
        # Math: The minimum radius needed for a square to fit in a circle
        # is half the square's diagonal.
        # Diagonal = width * sqrt(2)
        # Radius = Diagonal / 2
        return (self.peg.get_width() * math.sqrt(2)) / 2

### CLIENT CODE

In [7]:
def main():
    # A hole with radius 5
    hole = RoundHole(5)
    
    # 1. Standard Round Peg (Radius 5) -> FITS
    rpeg = RoundPeg(5)
    print(f"Round Peg (r=5) fits?  {hole.fits(rpeg)}")

    # 2. Square Peg (Width 5)
    small_sq_peg = SquarePeg(5)
    large_sq_peg = SquarePeg(10)

    # hole.fits(small_sq_peg) 
    # ^ CRASH! AttributeError: 'SquarePeg' object has no attribute 'get_radius'

    # 3. Use the Adapter!
    adapter_small = SquarePegAdapter(small_sq_peg)
    adapter_large = SquarePegAdapter(large_sq_peg)

    print(f"Square Peg (w=5) fits? {hole.fits(adapter_small)}") # True
    print(f"Square Peg (w=10) fits? {hole.fits(adapter_large)}") # False (Too big)

if __name__ == "__main__":
    main()

Round Peg (r=5) fits?  True
Square Peg (w=5) fits? True
Square Peg (w=10) fits? False


## Here is a clear, practical example of the Adapter Design Pattern in Python.

#### The Concept
The Adapter Pattern acts as a bridge between two incompatible interfaces. It allows classes to work together that couldn't otherwise because of incompatible method names or data formats.

#### Real-world Analogy
Traveling from the US to Europe. Your US laptop plug (Client) doesn't fit the European wall socket (Service). You need a Power Adapter to sit in the middle and translate the connection.

#### The Scenario: JSON vs XML
Imagine your modern Python application expects data in JSON format. However, you need to use a legacy 3rd-party Analytics Library that only outputs XML.

Instead of rewriting the entire legacy library (which might be impossible) or changing your entire app to support XML, you build an Adapter.

### THE TARGET (What our App expects)

In [8]:
from typing import Protocol

class ModernAnalytics(Protocol):
    """
    Our application is built to use this interface.
    It expects a simple dictionary (JSON-like) structure.
    """
    def analyze_data(self, data_json: str) -> None:
        ...

### THE ADAPTEE (The Incompatible Legacy Class)

In [9]:
import xml.etree.ElementTree as ET

class LegacyXMLAnalytics:
    """
    This is the old 3rd-party library we MUST use.
    Problem: It only accepts XML strings, not JSON.
    """
    def analyze_xml_data(self, xml_data: str) -> None:
        # Simulate complex processing
        root = ET.fromstring(xml_data)
        print(f"Legacy Lib: Analyzing XML -> User: {root.find('user').text}, "
              f"Event: {root.find('event').text}")

### THE ADAPTER

In [11]:
import json

class XMLAdapter:
    """
    The Adapter makes the Legacy class look like a Modern class.
    """
    def __init__(self, legacy_service: LegacyXMLAnalytics):
        self.legacy_service = legacy_service

    def analyze_data(self, data_json: str) -> None:
        """
        The key translation happens here:
        1. Receive JSON (from Client)
        2. Convert JSON to XML (for Adaptee)
        3. Call the Adaptee
        """
        print("Adapter: Converting JSON to XML...")
        
        # 1. Parse JSON
        data_dict = json.loads(data_json)
        
        # 2. Build XML string
        xml_data = (f"<root>"
                    f"<user>{data_dict['username']}</user>"
                    f"<event>{data_dict['action']}</event>"
                    f"</root>")
        
        # 3. Delegate to the legacy service
        self.legacy_service.analyze_xml_data(xml_data)

### CLIENT CODE

In [12]:
def client_code(analytics_system: ModernAnalytics):
    """
    The client code works with any class that follows the ModernAnalytics protocol.
    It doesn't know (or care) that it's actually talking to an XML library underneath.
    """
    sample_json = '{"username": "john_doe", "action": "clicked_button"}'
    analytics_system.analyze_data(sample_json)

def main():
    # 1. We have a Legacy Service
    old_system = LegacyXMLAnalytics()
    
    # 2. We wrap it in an Adapter
    adapter = XMLAdapter(old_system)
    
    # 3. We use it as if it were a modern system
    print("--- Client: Sending JSON Data ---")
    client_code(adapter)

if __name__ == "__main__":
    main()

--- Client: Sending JSON Data ---
Adapter: Converting JSON to XML...
Legacy Lib: Analyzing XML -> User: john_doe, Event: clicked_button
